**AUDIT NOTICE (integrity-audit-2026-07):** This notebook is out-of-sequence and cannot be used as a reproducibility record. The executable script  supersedes it for reproduction. This notebook is preserved as the historical record only. Do not re-run it to fix discrepancies; use the script.

In [31]:
import os
print(os.getcwd())

/Users/roshani/Downloads/cftr2_scraper


In [32]:
import re

def extract_variants(vcf_path):
    pattern = re.compile(r'p\.([A-Z][a-z]{2}\d+[A-Z][a-z]{2})')
    seen, variants = set(), []
    with open(vcf_path, encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            match = pattern.search(line)
            if match:
                name = match.group(1)
                if name not in seen:
                    seen.add(name)
                    variants.append(name)
    print(f"Found {len(variants)} variants")
    return variants

variants = extract_variants("../data/../data/All_Variants_VEP.Gene.vcf")
print(variants[:5])

Found 3220 variants
['Ser13Phe', 'Arg31Cys', 'Arg31Leu', 'Ser42Phe', 'Asp44Gly']


In [33]:
soup = BeautifulSoup(html, "html.parser")
print(soup.get_text()[:2000])

NameError: name 'BeautifulSoup' is not defined

In [34]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.get("https://cftr2.org/welcome")

# Accept the agreement if it appears
time.sleep(3)
print(driver.title)

Welcome to CFTR2 | CFTR2


In [35]:
# Find the search box and type the variant
search_box = driver.find_element(By.ID, "edit-mutation1")
search_box.send_keys("Ser13Phe")
time.sleep(2)

# Click submit
driver.execute_script("document.getElementById('search_submit').click()")
time.sleep(3)

print(driver.title)
print(driver.current_url)

NoSuchElementException: Message: no such element: Unable to locate element: {"method":"css selector","selector":"[id="edit-mutation1"]"}
  (Session info: chrome=148.0.7778.97); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
0   chromedriver                        0x0000000100b92584 cxxbridge1$str$ptr + 3225716
1   chromedriver                        0x0000000100b8a45c cxxbridge1$str$ptr + 3192652
2   chromedriver                        0x000000010064b8f4 _RNvCsiKAbIcglKMQ_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75152
3   chromedriver                        0x0000000100693fe4 _RNvCsiKAbIcglKMQ_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 371840
4   chromedriver                        0x00000001006d36e4 _RNvCsiKAbIcglKMQ_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 631680
5   chromedriver                        0x00000001006899a4 _RNvCsiKAbIcglKMQ_7___rustc35___r

In [ ]:
driver.execute_script("document.getElementById('edit-mutation1').value = 'Ser13Phe'")
driver.execute_script("document.getElementById('edit-mutation1').dispatchEvent(new Event('input'))")
time.sleep(4)

items = driver.find_elements(By.CSS_SELECTOR, "li.ui-menu-item")
print("Suggestions found:", len(items))
for item in items:
    print(item.text)

In [ ]:
driver.get("https://cftr2.org/mutations_history")
time.sleep(3)
print(driver.find_element(By.TAG_NAME, "body").text[:2000])

In [ ]:
import requests
import warnings
warnings.filterwarnings("ignore")

url = "https://cftr2.org/sites/default/files/CFTR2_30January2026.xlsx"
resp = requests.get(url, verify=False)

with open("../data/cftr2_variants.xlsx", "wb") as f:
    f.write(resp.content)

print("Downloaded!", len(resp.content), "bytes")

In [17]:
import pandas as pd

df = pd.read_excel("../data/cftr2_variants.xlsx")
print(df.shape)
print(df.columns.tolist())
print(df.head())

(2110, 9)
['List of current CFTR2 variants', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']
                   List of current CFTR2 variants Unnamed: 1 Unnamed: 2  \
0                           Date: 30 January 2026        NaN        NaN   
1            Number of patients in CFTR2: 122,935        NaN        NaN   
2     Number of variants reported in CFTR2: 2,092        NaN        NaN   
3  Number of variants with interpretations: 1,370        NaN        NaN   
4                               CF-causing: 1,245        NaN        NaN   

  Unnamed: 3 Unnamed: 4 Unnamed: 5 Unnamed: 6 Unnamed: 7 Unnamed: 8  
0        NaN        NaN        NaN        NaN        NaN        NaN  
1        NaN        NaN        NaN        NaN        NaN        NaN  
2        NaN        NaN        NaN        NaN        NaN        NaN  
3        NaN        NaN        NaN        NaN        NaN        NaN  
4        NaN        NaN        NaN        

In [36]:
print(df.iloc[5:15].to_string())

      legacy_name  protein_name                       cdna_name                                                 alt_names alleles_count allele_frequency            determination_2024            determination_2026 change protein_name_clean
5           R117H   p.Arg117His                        c.350G>A                                                    482G>A          2262         0.010715  Varying clinical consequence  Varying clinical consequence     No        p.Arg117His
6   3849+10kbC->T           p.?                  c.3718-2477C>T            c.3717+12191C>T, 3850-2477C->T, 3849+12191C->T          1990         0.009427                    CF-causing                    CF-causing     No                p.?
7       621+1G->T           p.?                      c.489+1G>T                                                       NaN          1860         0.008811                    CF-causing                    CF-causing     No                p.?
8           R553X     p.Arg553X             

In [37]:
df = pd.read_excel("../data/cftr2_variants.xlsx", header=10)
print(df.columns.tolist())
print(df.head(3))

['This detailed medical and genetics information is complicated and potentially confusing. We encourage you to discuss this information with your doctor, a genetic counselor, or a CF specialist. The information shown is for educational purposes only and is not intended for diagnostic use. You should not make any medical or reproductive decisions or change your health behavior based on this information without talking to your doctor.', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8']
  This detailed medical and genetics information is complicated and potentially confusing. We encourage you to discuss this information with your doctor, a genetic counselor, or a CF specialist. The information shown is for educational purposes only and is not intended for diagnostic use. You should not make any medical or reproductive decisions or change your health behavior based on this information without talking to your doctor.  \
0        

In [38]:
df = pd.read_excel("../data/cftr2_variants.xlsx", header=11)
df.columns = ['legacy_name', 'protein_name', 'cdna_name', 'alt_names', 
              'alleles_count', 'allele_frequency', 'determination_2024', 
              'determination_2026', 'change']
print(df.head(3))
print(df.shape)

  legacy_name protein_name       cdna_name                   alt_names  \
0     F508del  p.Phe508del  c.1521_1523del  c.1521_1523del, 1653delCTT   
1       G542X    p.Gly542X       c.1624G>T                     1756G>T   
2       G551D  p.Gly551Asp       c.1652G>A                     1784G>A   

  alleles_count allele_frequency determination_2024 determination_2026 change  
0        137363         0.650683         CF-causing         CF-causing     No  
1          5752         0.027247         CF-causing         CF-causing     No  
2          3831         0.018147         CF-causing         CF-causing     No  
(2099, 9)


In [39]:
# Match VCF variants to CFTR2 data
matched = df[df['legacy_name'].isin(variants)]
print(f"Matched {len(matched)} out of {len(variants)} variants")
print(matched[['legacy_name', 'protein_name', 'determination_2026']].head(10))

Matched 0 out of 3220 variants
Empty DataFrame
Columns: [legacy_name, protein_name, determination_2026]
Index: []


In [40]:
print("VCF variants:", variants[:5])
print("CFTR2 legacy names:", df['legacy_name'].tolist()[:5])
print("CFTR2 protein names:", df['protein_name'].tolist()[:5])

VCF variants: ['Ser13Phe', 'Arg31Cys', 'Arg31Leu', 'Ser42Phe', 'Asp44Gly']
CFTR2 legacy names: ['F508del', 'G542X', 'G551D', 'N1303K', 'W1282X']
CFTR2 protein names: ['p.Phe508del', 'p.Gly542X', 'p.Gly551Asp', 'p.Asn1303Lys', 'p.Trp1282X']


In [41]:
# Add p. prefix to VCF variants to match CFTR2 format
variants_with_p = ["p." + v for v in variants]

matched = df[df['protein_name'].isin(variants_with_p)]
print(f"Matched {len(matched)} out of {len(variants)} variants")
print(matched[['legacy_name', 'protein_name', 'determination_2026']].head(10))

Matched 656 out of 3220 variants
   legacy_name  protein_name            determination_2026
2        G551D   p.Gly551Asp                    CF-causing
3       N1303K  p.Asn1303Lys                    CF-causing
5        R117H   p.Arg117His  Varying clinical consequence
12      D1152H  p.Asp1152His  Varying clinical consequence
13        G85E    p.Gly85Glu                    CF-causing
15       R334W   p.Arg334Trp                    CF-causing
20       R347P   p.Arg347Pro                    CF-causing
23       L206W   p.Leu206Trp                    CF-causing
25       A455E   p.Ala455Glu                    CF-causing
28      R1066C  p.Arg1066Cys                    CF-causing


In [42]:
# Add p. prefix column to match
df['protein_name_clean'] = df['protein_name']

# Create result with all VCF variants, marking matched ones
import pandas as pd

vcf_df = pd.DataFrame({'variant': variants, 'protein_name': variants_with_p})
result = vcf_df.merge(df[['protein_name', 'legacy_name', 'determination_2026', 'allele_frequency']], 
                      on='protein_name', how='left')

result.to_csv("../data/cftr2_results.csv", index=False)
print(f"Saved! {len(result)} total variants")
print(f"Matched: {result['determination_2026'].notna().sum()}")
print(f"Not in CFTR2: {result['determination_2026'].isna().sum()}")

Saved! 3220 total variants
Matched: 656
Not in CFTR2: 2564


In [43]:
print(result['determination_2026'].value_counts())

determination_2026
No interpretation available     325
CF-causing                      226
Varying clinical consequence     72
Non CF-causing                   33
Name: count, dtype: int64
